# Aria on Google Colab

Free GPU, nothing to install locally. Run the cells top to bottom.

**Set the runtime to a GPU first:** *Runtime → Change runtime type → T4 GPU*.
It works on CPU too, just slower — the code picks whatever is available.

On a T4 the `small` preset trains in a few minutes and `base` (~100M
parameters) is realistic in an hour. Colab disconnects idle sessions, so the
last cell downloads everything Aria has learned to your own machine.

## 1. Set up

In [ ]:
!git clone -q https://github.com/Rocket-Ramuel/Aria-AI.git
%cd Aria-AI
!pip install -q -e .

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cpu':
    print('No GPU attached. Runtime -> Change runtime type -> T4 GPU for a much faster run.')

## 2. Build the corpus and tokenizer

Downloads ~19 MB of English prose, strips the markup and non-prose
appendices, trains a byte-level BPE vocabulary on it from scratch, and packs
the token streams. Takes a couple of minutes.

In [ ]:
!python -m aria prepare --vocab-size 8192 --block-size 256

## 3. Pretrain

This is resumable — re-run the cell to train further from where it stopped.
Watch the validation loss: falling means it is still learning, flat means
more steps will not help much and you want a bigger preset or more data.

On a T4, try `--preset base --batch-size 24 --steps 12000`.

In [ ]:
!python -m aria pretrain \
    --preset small \
    --steps 6000 \
    --batch-size 32 \
    --lr 6e-4 \
    --max-minutes 30 \
    --device $DEVICE

## 4. Talk to her — she learns while you do

Run the cell and type in the box that appears. Each turn prints what the
learner decided: the loss before and after, how surprising the exchange was,
how far the weights have drifted from their anchor, and whether anything was
rolled back.

Useful things to type:

| | |
|---|---|
| `/correct <what she should have said>` | highest-value signal; bypasses the surprise gate |
| `/status` | what the learner has been doing |
| `/memory` | what she has remembered |
| `/consolidate` | fold the learning into the real weight matrices now |
| `quit` | leave the loop (memory is saved) |

In [ ]:
import torch
from aria.chat import ChatSession, _command

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
session = ChatSession('runs/aria/base.pt', device=DEVICE, verbose=True)
print(f'Aria — {session.model.num_params()/1e6:.2f}M parameters, learning on.\n')

while True:
    try:
        line = input('you> ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not line:
        continue
    if line in ('quit', 'exit', '/quit', '/exit'):
        break
    if line.startswith('/'):
        _command(session, line)
        continue
    reply, report = session.turn(line)
    print(f'aria> {reply}')
    if report is not None:
        print(f'      {report.line()}\n')

session.save()
print('saved.')

## 5. Check that the learning is real

Measures four things against the model you just trained: that teaching lowers
the loss on what was taught, that the lesson survives 40 unrelated new ones,
that held-out English does not degrade, and that it all survives a restart.

In [ ]:
!python scripts/demo_learning.py --device $DEVICE

## 6. Take it home

Colab wipes the machine when the session ends. This packages the weights, the
tokenizer and everything Aria learned into one file and downloads it.

Unpack it into a local clone and carry on with `aria serve` — no GPU needed to
talk to her.

In [ ]:
!tar czf aria_trained.tar.gz runs/aria data/tokenizer.json data/meta.json
!ls -lh aria_trained.tar.gz

from google.colab import files
files.download('aria_trained.tar.gz')